# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides an end-to-end demonstration for loading and exploring the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL, allowing programmatic access to both its rich metadata and the tabular data itself.

_Dataset DOI_: [10.71728/senscience.qs2f-h81p](https://sen.science/doi/10.71728/senscience.qs2f-h81p)


In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant
!pip install -q matplotlib seaborn

## 1. Data Loading
Load the metadata and available record sets from the dataset using `mlcroissant`, and display key dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access main metadata as a native object
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Publication Date: {getattr(metadata, 'datePublished', '')}")


## 2. Data Overview
Let's inspect available record sets, their `@id`s, and corresponding fields and columns. 

Using the `mlcroissant` API, we can print out record sets and their contained fields/columns (all referenced by their `@id`).

In [ ]:
# List all available record sets by @id and show their fields and columns @id's.
from mlcroissant.types.records import _RecordSet

def get_record_sets(ds):
    """Extract all RecordSet @id's and their fields/columns."""
    record_sets_info = []
    for rset in getattr(ds.metadata, 'recordSet', []):
        rset_id = getattr(rset, '@id', None)
        rset_name = getattr(rset, 'name', None)
        fields = []
        if hasattr(rset, 'field') and rset.field:
            for field in rset.field:
                fid = getattr(field, '@id', None)
                fname = getattr(field, 'name', None)
                fields.append((fid, fname))
        columns = []
        if hasattr(rset, 'column') and rset.column:
            for col in rset.column:
                cid = getattr(col, '@id', None)
                cname = getattr(col, 'name', None)
                columns.append((cid, cname))
        record_sets_info.append({'@id': rset_id, 'name': rset_name, 'fields': fields, 'columns': columns})
    return record_sets_info

record_sets_info = get_record_sets(dataset)

# Print out summary of record sets and their fields/columns (referenced by @id)
for rs in record_sets_info:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs['name']}")
    if rs['fields']:
        print("  Fields:")
        for fid, fname in rs['fields']:
            print(f"    {fid} (name: {fname})")
    if rs['columns']:
        print("  Columns:")
        for cid, cname in rs['columns']:
            print(f"    {cid} (name: {cname})")
    print()

# For cases where the dataset inlines all data into a single main record set, print a basic sample of records:
main_record_set_id = None
if record_sets_info:
    main_record_set_id = record_sets_info[0]['@id']
    print(f"\nSample records from RecordSet {main_record_set_id}:")
    records = dataset.records(record_set=main_record_set_id)
    for i, rec in enumerate(records):
        if i >= 2:
            break
        print(rec)


## 3. Data Extraction
We now extract the records from available record sets into Pandas DataFrames for further exploration.

Entities (columns, fields) and the record set are referenced strictly by their `@id`, as required for programmatic reproducibility.

In [ ]:
# Prepare list of record set @id's discovered
record_set_ids = [info['@id'] for info in record_sets_info]

dataframes = {}
for record_set_id in record_set_ids:
    rows = list(dataset.records(record_set=record_set_id))
    # Only non-empty dataframes
    if rows:
        dataframes[record_set_id] = pd.DataFrame(rows)

# Show available columns for the principal table (main record set)
if main_record_set_id in dataframes:
    print(f"Columns in RecordSet '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set found with records.")

## 4. Exploratory Data Analysis (EDA)
Let us demonstrate some processing steps: filtering, normalization, and grouping using `@id`-referenced fields. 

_Note: Please refer to the printed list above for valid field `@id`s/column names._

In [ ]:
# Choose the main record set and examine field @id's
df = dataframes.get(main_record_set_id)
if df is not None:
    print(f"Columns (@id) in chosen RecordSet: {df.columns.tolist()}")
else:
    raise Exception(f"No DataFrame loaded for record set {main_record_set_id}")

# === Example EDA: filter, normalize, group by ===
# Pick a numeric field @id. Adjust the name as per dataset -- for demo, try common ones:
possible_numeric_ids = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower()]
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]

# Print value distribution
print(f"\nDistribution of {numeric_field_id} before filtering:")
print(df[numeric_field_id].describe())

# Filter records (e.g., age or interval > median) -- using threshold demonstration
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nSample normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a relevant categorical field (e.g. sex, anatomical location);
    # Try to detect a categorical field @id
    group_field_id = None
    for c in df.columns:
        if 'sex' in c.lower() or 'site' in c.lower() or 'location' in c.lower() or 'stage' in c.lower():
            group_field_id = c
            break
    if not group_field_id:
        # fallback
        group_field_id = df.select_dtypes(include='object').columns[0]

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Column '{numeric_field_id}' is not numeric.")


## 5. Visualization
Visualize distributions and group comparisons for selected fields. All fields are referenced via their `@id`, as above.


In [ ]:
# Plot histogram and boxplot for the analyzed numeric field, grouped by categorical field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot: DataFrame or fields not available.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to programmatically:
- Load a clinical dataset according to its Croissant schema
- Inspect all record sets and find their fields, referencing all metadata and data by `@id`
- Extract, process, and visualize tabular data from record sets into Pandas DataFrames
- Demonstrate basic data filtering, normalization, aggregation, and visualization using only Croissant-provided IDs

This reproducible approach enables transparent, schema-linked exploration crucial for FAIR and responsible biomedical data sharing. For further work, you may extend this notebook to perform analysis specific to clinical outcomes, survival, or biomarker prediction.